## Adding layers for event phrase main words

Initially, event layers for all event phrase words without semantic classes, with semantic classes, and with aggregated semantic classes were added to both Estonian TimeML and temporal facts corpora. 

Now, based on these initial layers, the same layers for event phrase main words are added to both corpora. Event phrase main words are found recursively from Stanza syntax tree of sentences.

In [ ]:
import os

import estnltk
from estnltk import Text
from estnltk_neural.taggers import StanzaSyntaxTagger
from corpus_methods.file_operations import *

#### Initializing StanzaSyntaxTagger

In [150]:
stanza_tagger = StanzaSyntaxTagger(input_type='morph_analysis', input_morph_layer='morph_analysis',
                                   add_parent_and_children=True)

#### Methods

In [ ]:
# finds event phrases of given sentence
def get_sentence_phrases(text_obj, sentence):
    phrase_matches = []
    for i in range(len(text_obj.gold_event_phrases)):
        phrase = []
        for j in range(len(text_obj.gold_event_phrases[i])):
            for word in sentence.words:
                if word == text_obj.gold_event_phrases[i][j]:
                    phrase.append(word)
        if len(phrase) > 0:
            phrase_matches.append(phrase)
    return phrase_matches


# gets event phrases' main word
def get_sentence_phrase_main_word(phrase, text_obj, current_word):
    current_word_span = text_obj.words.get(current_word)
    if current_word_span in phrase:
        return current_word_span
    else:
        for child in current_word.children:
            #print(text_obj.words.get(current_word))
            #print(f"current word: {current_word}, child: {current_word.children[i]}")
            current_word_span = get_sentence_phrase_main_word(phrase, text_obj, child)
            if current_word_span in phrase:
                return current_word_span

In [ ]:
def create_event_main_word_layer(text_obj, event_layer_name):
    gold_word_events_main = estnltk.Layer(name="gold_word_events_main", text_object=text_obj, attributes=['nertag'],
                                     enveloping='words')  
    all_events_main_words = []
    for sentence in text_obj.sentences:
        sent_phrases = get_sentence_phrases(text_obj, sentence)
        sentence_root = None
        for j in range(len(sentence.stanza_syntax)):
            if sentence.stanza_syntax.deprel[j] == 'root':
                sentence_root = sentence.stanza_syntax[j]           
        sent_events_main_words = []
        for phrase in sent_phrases:
            if len(phrase) == 1:
                sent_events_main_words.append(phrase[0])
            else:
                sent_events_main_words.append(get_sentence_phrase_main_word(phrase, text_obj, sentence_root))
        assert len(sent_events_main_words) == len(sent_phrases), "Different number of events and main words"
        all_events_main_words += sent_events_main_words
                
    for i in range(len(text_obj.gold_word_events)):
        iob_word = text_obj.gold_word_events[i]
        event = None
        if iob_word['nertag'] == 'B-EVENT' or iob_word['nertag'] == 'I-EVENT' and iob_word in all_events_main_words:
            event = text_obj[event_layer_name].get(iob_word)
        if event:
            gold_word_events_main.add_annotation([iob_word[0].base_span], nertag=iob_word['nertag'])
        else:
            gold_word_events_main.add_annotation([iob_word[0].base_span], nertag='O')
    
    return gold_word_events_main


def create_event_main_word_w_classes_layer(text_obj, event_layer_name, class_attr_name):
    gold_word_events_main_w_classes = estnltk.Layer(name="gold_word_events_main_w_classes", text_object=text_obj, attributes=['nertag'],
                                     enveloping='words')
    all_events_main_words = []
    for sentence in text_obj.sentences:
        sent_phrases = get_sentence_phrases(text_obj, sentence)
        sentence_root = None
        for j in range(len(sentence.stanza_syntax)):
            if sentence.stanza_syntax.deprel[j] == 'root':
                sentence_root = sentence.stanza_syntax[j]           
        sent_events_main_words = []
        for phrase in sent_phrases:
            if len(phrase) == 1:
                sent_events_main_words.append(phrase[0])
            else:
                sent_events_main_words.append(get_sentence_phrase_main_word(phrase, text_obj, sentence_root))
        assert len(sent_events_main_words) == len(sent_phrases), "Different number of events and main words"
        all_events_main_words += sent_events_main_words
                
    for i in range(len(text_obj.gold_word_events)):
        iob_word = text_obj.gold_word_events[i]
        event = None
        if iob_word['nertag'] == 'B-EVENT' or iob_word['nertag'] == 'I-EVENT' and iob_word in all_events_main_words:
            event = text_obj[event_layer_name].get(iob_word)
        if event:
            event_class = event[class_attr_name]
            gold_word_events_main_w_classes.add_annotation([iob_word[0].base_span], nertag=iob_word['nertag']+'_'+event_class) # Eesti TimeML korpuse puhul event_class[0]
        else:
            gold_word_events_main_w_classes.add_annotation([iob_word[0].base_span], nertag='O')
    
    return gold_word_events_main_w_classes


def create_event_main_word_w_agg_classes_layer(text_obj, event_layer_name, class_attr_name):
    agg_mapping = {'REPORTING': 'ARG_CLASS',
               'PERCEPTION': 'ARG_CLASS',
               'ASPECTUAL': 'ARG_CLASS',
               'I_ACTION': 'ARG_CLASS',
               'I_STATE': 'ARG_CLASS',
               'STATE': 'STATE',
               'MODAL': 'MODAL',
               'OCCURRENCE': 'OCCURRENCE'}
    gold_word_events_main_w_agg_classes = estnltk.Layer(name="gold_word_events_main_w_agg_classes", text_object=text_obj, attributes=['nertag'],
                                     enveloping='words')   
    all_events_main_words = []
    for sentence in text_obj.sentences:
        sent_phrases = get_sentence_phrases(text_obj, sentence)
        sentence_root = None
        for j in range(len(sentence.stanza_syntax)):
            if sentence.stanza_syntax.deprel[j] == 'root':
                sentence_root = sentence.stanza_syntax[j]           
        sent_events_main_words = []
        for phrase in sent_phrases:
            if len(phrase) == 1:
                sent_events_main_words.append(phrase[0])
            else:
                sent_events_main_words.append(get_sentence_phrase_main_word(phrase, text_obj, sentence_root))
        assert len(sent_events_main_words) == len(sent_phrases), "Different number of events and main words"
        all_events_main_words += sent_events_main_words
                
    for i in range(len(text_obj.gold_word_events)):
        iob_word = text_obj.gold_word_events[i]
        event = None
        if iob_word['nertag'] == 'B-EVENT' or iob_word['nertag'] == 'I-EVENT' and iob_word in all_events_main_words:
            event = text_obj[event_layer_name].get(iob_word)
        if event:
            event_class = event[class_attr_name]
            gold_word_events_main_w_agg_classes.add_annotation([iob_word[0].base_span], nertag=iob_word['nertag']+'_'+agg_mapping[event_class]) # Eesti TimeML korpuse puhul agg_mapping[event_class[0]]
        else:
            gold_word_events_main_w_agg_classes.add_annotation([iob_word[0].base_span], nertag='O')
    
    return gold_word_events_main_w_agg_classes


def create_event_main_word_w_agg_classes_no_modal_layer(text_obj, event_layer_name, class_attr_name):
    agg_mapping = {'REPORTING': 'ARG_CLASS',
               'PERCEPTION': 'ARG_CLASS',
               'ASPECTUAL': 'ARG_CLASS',
               'I_ACTION': 'ARG_CLASS',
               'I_STATE': 'ARG_CLASS',
               'STATE': 'STATE',
               'MODAL': 'ARG_CLASS',
               'OCCURRENCE': 'OCCURRENCE'}
    gold_word_events_main_w_agg_classes_no_modal = estnltk.Layer(name="gold_word_events_main_w_agg_classes_no_modal", text_object=text_obj, attributes=['nertag'],
                                     enveloping='words')   
    all_events_main_words = []
    for sentence in text_obj.sentences:
        sent_phrases = get_sentence_phrases(text_obj, sentence)
        sentence_root = None
        for j in range(len(sentence.stanza_syntax)):
            if sentence.stanza_syntax.deprel[j] == 'root':
                sentence_root = sentence.stanza_syntax[j]           
        sent_events_main_words = []
        for phrase in sent_phrases:
            if len(phrase) == 1:
                sent_events_main_words.append(phrase[0])
            else:
                sent_events_main_words.append(get_sentence_phrase_main_word(phrase, text_obj, sentence_root))
        assert len(sent_events_main_words) == len(sent_phrases), "Different number of events and main words"
        all_events_main_words += sent_events_main_words
                
    for i in range(len(text_obj.gold_word_events)):
        iob_word = text_obj.gold_word_events[i]
        event = None
        if iob_word['nertag'] == 'B-EVENT' or iob_word['nertag'] == 'I-EVENT' and iob_word in all_events_main_words:
            event = text_obj[event_layer_name].get(iob_word)
        if event:
            event_class = event[class_attr_name]
            gold_word_events_main_w_agg_classes_no_modal.add_annotation([iob_word[0].base_span], nertag=iob_word['nertag']+'_'+agg_mapping[event_class]) # Eesti TimeML korpuse puhul agg_mapping[event_class[0]]
        else:
            gold_word_events_main_w_agg_classes_no_modal.add_annotation([iob_word[0].base_span], nertag='O')
    
    return gold_word_events_main_w_agg_classes_no_modal

### Adding event main word layers to Estonian TimeML Corpus

In [ ]:
# -- source and target folder path
json_dir_path = 'processed_data/EstTimeML_corpus_json/'

n_saved_articles = 0
for filename in os.listdir(json_dir_path):
    print(filename)
    
    # load Text-object
    text_obj = load_Text_from_json(json_dir_path, filename)
    # add stanza syntax layer
    stanza_tagger.tag(text_obj)
    # add new layers to Text-object
    text_obj.add_layer( create_event_main_word_layer(text_obj, 'gold_events') )
    text_obj.add_layer( create_event_main_word_w_classes_layer(text_obj, 'gold_events', 'event_class') )
    text_obj.add_layer( create_event_main_word_w_agg_classes_layer(text_obj, 'gold_events', 'event_class') )
    text_obj.add_layer( create_event_main_word_w_agg_classes_no_modal_layer(text_obj, 'gold_events', 'event_class') )
    
    # save Text-object in JSON format
    save_Text_to_json(json_dir_path, text_obj)
    n_saved_articles+=1
    
print()
print(f"Saved {n_saved_articles} articles in JSON-format.")

aja_ml_2002_47.tasak.a006.sol.json
aja_ml_2002_47.tasak.a007.sol.json
aja_ml_2002_47.tasak.a008.sol.json
aja_ml_2002_47.tasak.a014.sol.json
aja_ml_2002_47.tasak.a016.sol.json
aja_ml_2002_47.tasak.a017.sol.json
aja_ml_2002_47.tasak.a023.sol.json
aja_ml_2002_47.tasak.a025.sol.json
aja_ml_2002_47.tasak.a028.sol.json
aja_ml_2002_47.tasak.a030.sol.json
aja_ml_2002_47.tasak.a031.sol.json
aja_ml_2002_47.tasak.a033.sol.json
aja_ml_2002_47.tasak.a038.sol.json
aja_ml_2002_47.tasak.a040.sol.json
aja_ml_2002_47.tasak.a041.sol.json
aja_ml_2002_47.tasak.a042.sol.json
aja_ml_2002_47.tasak.a045.sol.json
aja_ml_2002_47.tasak.a051.sol.json
aja_pm_2000_10_04.tasak.a001.sol.json
aja_pm_2000_10_04.tasak.a003.sol.json
aja_pm_2000_10_04.tasak.a004.sol.json
aja_pm_2000_10_04.tasak.a006.sol.json
aja_pm_2000_10_04.tasak.a007.sol.json
aja_pm_2000_10_04.tasak.a009.sol.json
aja_pm_2000_10_04.tasak.a010.sol.json
aja_pm_2000_10_04.tasak.a012.sol.json
aja_pm_2000_10_04.tasak.a013.sol.json
aja_pm_2000_10_04.tasak.a015

In [ ]:
# example of removing layers (if necessary) (commented out)
# reading from JSON-files
#json_dir_path = 'processed_data/EstTimeML_corpus_json/'

#for filename in os.listdir(json_dir_path):
#    text_obj = load_Text_from_json(json_dir_path, filename)
#    text_obj.pop_layer('gold_word_events_main')
#    text_obj.pop_layer('gold_word_events_main_w_classes')
#    text_obj.pop_layer('gold_word_events_main_w_agg_classes')
#    text_obj.pop_layer('gold_word_events_main_w_agg_classes_no_modal')
    # save Text-object in JSON format
#    save_Text_to_json(json_dir_path, text_obj)

### Adding event main word layers to temporal facts corpus

In [ ]:
json_dir_path = 'processed_data/temporal_facts_corpus_json/'

n_saved_articles = 0
for filename in os.listdir(json_dir_path):
    print(filename)
    
    # load Text-object
    text_obj = load_Text_from_json(json_dir_path, filename)
    stanza_tagger.tag(text_obj)
    # add new layers to Text-object
    text_obj.add_layer( create_event_main_word_layer(text_obj, 'events') )
    text_obj.add_layer( create_event_main_word_w_classes_layer(text_obj, 'events', 'class') )
    text_obj.add_layer( create_event_main_word_w_agg_classes_layer(text_obj, 'events', 'class') )
    text_obj.add_layer( create_event_main_word_w_agg_classes_no_modal_layer(text_obj, 'events', 'class') )
    
    # save Text-object in JSON format
    save_Text_to_json(json_dir_path, text_obj)
    n_saved_articles+=1
    
print()
print(f"Saved {n_saved_articles} articles in JSON-format.")

aja-arvamus_EPL_1998_06_12.json
aja-arvamus_EPL_1999_09_11.json
aja-arvamus_EPL_2000_03_28.json
aja-arvamus_EPL_2000_08_19.json
aja-arvamus_EPL_2003_03_07.json
aja-arvamus_EPL_2007_05_15.json
aja-arvamus_luup_1996_15.json
aja-arvamus_luup_1996_21.json
aja-arvamus_pm_1995_12_07.json
aja-arvamus_pm_1996_07_10.json
aja-arvamus_pm_1997_04_17.json
aja-arvamus_pm_2000_10_02.json
aja-eesti_EPL_1996_12_11.json
aja-eesti_EPL_1997_01_22.json
aja-eesti_EPL_1999_07_24.json
aja-eesti_EPL_1999_07_31.json
aja-eesti_EPL_2000_03_28.json
aja-eesti_EPL_2000_08_19.json
aja-eesti_EPL_2001_06_02_1.json
aja-eesti_EPL_2001_06_02_2.json
aja-eesti_EPL_2002_03_25.json
aja-eesti_EPL_2003_05_02.json
aja-eesti_EPL_2004_04_30.json
aja-eesti_EPL_2005_11_05.json
aja-eesti_EPL_2006_02_25.json
aja-eesti_EPL_2007_04_11.json
aja-eesti_luup_1996_20.json
aja-eesti_luup_1998_24.json
aja-eesti_pm_1995_11_29.json
aja-eesti_pm_1995_12_20.json
aja-eesti_pm_1997_04_08.json
aja-eesti_pm_1997_07_17.json
aja-eesti_pm_1998_05_07.tasa